# Financial Report Standardization & Cleaning Pipeline

This notebook automates the loading, cleaning, and standardization of quarterly financial report data for Israeli companies, from q1 2020 through full-year 2024. It reads raw JSON reports from Google Drive, unifies similar company names using fuzzy matching and manual rules, and maps them to official tickers using an Excel reference. The data is deduplicated by selecting the most complete entry per company and saved back to Drive for further analysis. While the paths reference a specific q1 2021 example, the same pipeline is applied across all time periods consistently.


### Imports:

In [387]:
!pip install rapidfuzz

In [388]:
import pandas as pd
import numpy as np
import json

from rapidfuzz import process, fuzz
from collections import defaultdict


import re

import ast

### Mounting to drive:

In [389]:
# connect to drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Reading & presenting example file:

In [390]:
# read json file from path /content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/financial_reports_q1_2021.json with utf-8
with open('/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/financial_reports_q1_2021.json', 'r', encoding='utf-8') as f:
    financial_reports_q1_2021 = json.load(f)

In [391]:
financial_reports_q1_2021

[{'Company Name': 'Accel Solutions Group Ltd',
  'Report Date': '2025-04-16',
  'Revenue': 'N/A',
  'Net Income': 'N/A',
  'Earnings Per Share (EPS)': 0.033,
  'Gross Profit': 'N/A',
  'Operating Income (EBIT)': 'N/A',
  'Operating Cash Flow': 'N/A',
  'Investing Cash Flow': 'N/A',
  'Financing Cash Flow': 'N/A',
  'Total Assets': 'N/A',
  'Total Liabilities': 'N/A',
  'Short-Term Debt': 'N/A',
  'Long-Term Debt': 'N/A',
  'Shareholders’ Equity': 'N/A',
  'Dividends Paid': 'N/A',
  'Number of Outstanding Shares': 164755008,
  'Guidance/Forecast': 'N/A',
  'Key Performance Indicators (KPIs)': 'N/A'},
 {'Company Name': 'Ackerstein Group Ltd',
  'Report Date': '2025-04-16',
  'Revenue': 902353000,
  'Net Income': 123406000,
  'Earnings Per Share (EPS)': 0.43,
  'Gross Profit': 250157000,
  'Operating Income (EBIT)': 110407000,
  'Operating Cash Flow': 'N/A',
  'Investing Cash Flow': 'N/A',
  'Financing Cash Flow': 'N/A',
  'Total Assets': 1944990000,
  'Total Liabilities': 'N/A',
  'Short

### Data Loading & Initial Cleaning

This function dynamically loads quarterly financial report JSON files from Google Drive, flattens nested structures, and cleans the numeric columns by removing formatting and converting values to floats. It ensures consistency across reports by standardizing the parameters per file into columns and preserving key textual fields for further processing.


In [392]:


def load_and_clean_financial_report(year: int, quarter: str):
    # Step 1: Build the file path dynamically
    path = f"/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/financial_reports_{quarter}_{year}.json"

    # Step 2: Load JSON
    with open(path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    # Step 3: Flatten list of lists
    flattened_data = []
    for item in raw_data:
        if isinstance(item, list):
            flattened_data.extend(item)
        else:
            flattened_data.append(item)

    # Step 4: Convert to DataFrame
    df = pd.DataFrame(flattened_data)

    # Step 5: Keep only first 19 columns
    df = df.iloc[:, :19]

    # Step 6: Replace 'N/A' with NaN
    df = df.replace('N/A', np.nan)

    # Step 7: Clean all columns except for preserved string ones
    string_columns = ['Company Name', 'Report Date', 'Guidance/Forecast']

    for col in df.columns:
        if col not in string_columns:
            # Clean number formatting and convert to float
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(',', '', regex=False)
                .str.replace(r'\((.*?)\)', r'-\1', regex=True)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Step 8: Dynamically name the DataFrame
    var_name = f"financial_reports_{quarter}_{year}"
    globals()[var_name] = df
    return df


In [393]:
df2021_q1 = load_and_clean_financial_report(2021, "q1")

<ipython-input-392-7ff2bb4ba0f6>:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('N/A', np.nan)


In [394]:
df2021_q1

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,Accel Solutions Group Ltd,2025-04-16,NaN,NaN,0.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164755008.0,NaN,NaN
1,Ackerstein Group Ltd,2025-04-16,9.023530e+08,123406000.0,0.430,2.501570e+08,1.104070e+08,NaN,NaN,NaN,1.944990e+09,NaN,NaN,7.484000e+07,NaN,NaN,288060992.0,NaN,NaN
2,Kvutzat Acro Ltd,2025-04-16,7.484870e+08,20589000.0,0.330,2.003150e+08,9.604400e+07,NaN,NaN,NaN,6.929554e+09,NaN,NaN,3.020661e+09,NaN,NaN,63043600.0,NaN,NaN
3,Adgar Investments and Development Ltd,2025-04-16,3.380300e+08,-7128000.0,-0.040,2.888100e+08,2.350480e+08,NaN,NaN,NaN,5.490045e+09,NaN,NaN,2.967060e+09,NaN,NaN,164856000.0,NaN,NaN
4,Aerodrome Group Ltd,2025-04-16,1.445000e+07,-22798000.0,-0.250,9.900000e+05,-1.356500e+07,NaN,NaN,NaN,3.909100e+07,NaN,NaN,3.510000e+05,NaN,NaN,91569504.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,Zanlakol Ltd,2025-04-16,4.815950e+08,49839000.0,3.490,1.544380e+08,8.245000e+07,NaN,NaN,NaN,4.842290e+08,NaN,NaN,0.000000e+00,NaN,NaN,13991100.0,NaN,NaN
450,Zephyrus Wing Energies Ltd,2025-04-16,1.967430e+08,82409000.0,1.270,1.571700e+08,8.147100e+07,NaN,NaN,NaN,1.492863e+09,NaN,NaN,7.211830e+08,NaN,NaN,65017300.0,NaN,NaN
451,Z.M.H Hammerman Ltd,2025-04-16,3.900150e+08,39988000.0,2.000,4.439500e+07,3.852000e+06,NaN,NaN,NaN,1.805569e+09,NaN,NaN,2.282560e+08,NaN,NaN,20022000.0,NaN,NaN
452,ZOOZ Power Ltd.,2025-04-16,1.041000e+06,-10990000.0,-4.250,-4.860000e+05,-1.053600e+07,NaN,NaN,NaN,1.283700e+07,NaN,NaN,NaN,NaN,NaN,12105500.0,NaN,NaN


### Fuzzy Matching for Company Name Unification

This section uses fuzzy string matching to automatically group and unify variations of company names that refer to the same entity. It selects the most complete row (fewest missing values) from each group and assigns a canonical name, reducing redundancy in the dataset.


In [395]:


def auto_group_similar_names(df, threshold=80):
    company_names = df['Company Name'].dropna().unique()
    grouped = defaultdict(list)
    used = set()

    for name in company_names:
        if name in used:
            continue
        # Find close matches
        matches = process.extract(name, company_names, scorer=fuzz.token_sort_ratio)
        group = [match for match, score, _ in matches if score >= threshold]
        for match in group:
            used.add(match)
        grouped[name].extend(group)

    return grouped

def unify_companies_by_fuzzy_match(df, threshold=80):
    name_mapping = auto_group_similar_names(df, threshold)
    unified_rows = []

    for canonical_name, variations in name_mapping.items():
        subset = df[df['Company Name'].isin(variations)]
        if subset.empty:
            continue
        unified = subset.bfill().iloc[0]
        unified['Company Name'] = canonical_name
        unified_rows.append(unified)

    return pd.DataFrame(unified_rows)

# 👇 Run this
df_unified = unify_companies_by_fuzzy_match(df2021_q1, threshold=80)


In [396]:
df_unified

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,Accel Solutions Group Ltd,2025-04-16,NaN,NaN,0.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164755008.0,NaN,NaN
1,Ackerstein Group Ltd,2025-04-16,9.023530e+08,123406000.0,0.430,2.501570e+08,1.104070e+08,NaN,NaN,NaN,1.944990e+09,NaN,NaN,7.484000e+07,NaN,NaN,288060992.0,NaN,NaN
2,Kvutzat Acro Ltd,2025-04-16,7.484870e+08,20589000.0,0.330,2.003150e+08,9.604400e+07,NaN,NaN,NaN,6.929554e+09,NaN,NaN,3.020661e+09,NaN,NaN,63043600.0,NaN,NaN
3,Adgar Investments and Development Ltd,2025-04-16,3.380300e+08,-7128000.0,-0.040,2.888100e+08,2.350480e+08,NaN,NaN,NaN,5.490045e+09,NaN,NaN,2.967060e+09,NaN,NaN,164856000.0,NaN,NaN
4,Aerodrome Group Ltd,2025-04-16,1.445000e+07,-22798000.0,-0.250,9.900000e+05,-1.356500e+07,NaN,NaN,NaN,3.909100e+07,NaN,NaN,3.510000e+05,NaN,NaN,91569504.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,Zanlakol Ltd,2025-04-16,4.815950e+08,49839000.0,3.490,1.544380e+08,8.245000e+07,NaN,NaN,NaN,4.842290e+08,NaN,NaN,0.000000e+00,NaN,NaN,13991100.0,NaN,NaN
450,Zephyrus Wing Energies Ltd,2025-04-16,1.967430e+08,82409000.0,1.270,1.571700e+08,8.147100e+07,NaN,NaN,NaN,1.492863e+09,NaN,NaN,7.211830e+08,NaN,NaN,65017300.0,NaN,NaN
451,Z.M.H Hammerman Ltd,2025-04-16,3.900150e+08,39988000.0,2.000,4.439500e+07,3.852000e+06,NaN,NaN,NaN,1.805569e+09,NaN,NaN,2.282560e+08,NaN,NaN,20022000.0,NaN,NaN
452,ZOOZ Power Ltd.,2025-04-16,1.041000e+06,-10990000.0,-4.250,-4.860000e+05,-1.053600e+07,NaN,NaN,NaN,1.283700e+07,NaN,NaN,NaN,NaN,NaN,12105500.0,NaN,NaN


In [397]:


# Load the mapping Excel file (ticker → company name)
mapping_path = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Noam's work/Best model per stock/Helpful files /company_name_to_ticker.xlsx"
mapping_df = pd.read_excel(mapping_path)
company_map = mapping_df[['Ticker', 'CompanyName']].dropna()
reference_names = company_map['CompanyName'].str.upper().unique().tolist()

# Manual name override dictionary
manual_name_map = {
    k.upper(): v.upper()
    for k, v in {
        "A. Luzon Real Estate and Finance": "LUZON GROUP",
        "Alon Blue Square Israel": "BLUE SQUARE REAL ESTATE",
        "Accel Solutions Group": "ACCEL",
        "Electreon Wireless": "ELECTREON",
        "El Al Israel Airlines.": "EL AL",
        "Elbit Medical Technologies": "ELBIT MEDITEC",
        "Enlight Renewable Energy": "ENLIGHT ENERGY",
        "Enlivex Therapeutics": "ENLIVEX",
        "Alony Hetz Properties and Investments": "ALONY HETZ",
        "Aylon Bituach Hanpakot Vegiusi Hon LTD": "AYALON HOLD.",
        "Avgol Industries 1953": "AVGOL",
        "Bet Shemesh Engines Holdings (1997) LTD.": "BET SHEMESH",
        "Bicuri Sadeh": "BIKUREY HASHDE",
        "Bikurei Sadeh (Holdings)": "BIKUREY HASHDE",
        "Biolight Life Sciences": "BIOLIGHT",
        "C.I. Systems (Israel)": "C I SYSTEMS",
        "Cellcom Israel": "CELLCOM",
        "Clal Insurance Enterprises Holdings": "CLAL INSURANCE",
        "Delek Motors Vehicle Systems": "DELEK AUTOMOTIV"

    }.items()
}

GENERIC_SUFFIXES = ['HOLDINGS', 'INVESTMENTS', 'PROPERTIES', 'GROUP', 'REAL ESTATE', 'COMPANY', 'PROPERTIES AND INVESTMENTS', 'SOLUTIONS GROUP',
                    'LIFE SCIENCES','REAL ESTATE', 'REAL ESTATE AND FINANCE', 'INDUSTRIAL DEVELOPMENT', 'ENGINEERING','COMPUTING', 'INTERNET']

def generate_variants(name: str):
    """
    Generate multiple variants of a cleaned name:
    - base form
    - base + each generic suffix
    - base - each generic word (if it exists)
    """
    base = clean_name(name)
    variants = set([base])

    # If base already contains suffixes, try removing them
    words = base.split()
    without_suffix = ' '.join([w for w in words if w not in GENERIC_SUFFIXES])
    variants.add(without_suffix)

    # Try adding common suffixes
    for suffix in GENERIC_SUFFIXES:
        variants.add(f"{base} {suffix}")
        variants.add(f"{without_suffix} {suffix}")

    return variants


# Clean helper
def clean_name(name: str) -> str:
    name = name.upper()
    name = re.sub(r'\b(LTD.|INC|LTD|CORP|LIMITED|\.|,|\&)\b', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Fuzzy matcher using cleaned names
def match_company_name_fuzzy(name: str, reference_names, threshold=85):
    # 1. Manual override first
    name_cleaned = clean_name(name)
    if name_cleaned in manual_name_map:
        return manual_name_map[name_cleaned]

    # 2. Clean all reference names into a searchable map
    ref_cleaned_map = {clean_name(ref): ref for ref in reference_names}

    # 3. Try all name variants
    name_variants = generate_variants(name)
    best_match, best_score = None, -1

    for variant in name_variants:
        match, score, _ = process.extractOne(variant, list(ref_cleaned_map.keys()), scorer=fuzz.token_sort_ratio)
        if score > best_score and score >= threshold:
            best_score = score
            best_match = ref_cleaned_map[match]

    return best_match


# Apply to your unified DataFrame
def standardize_company_names(df, reference_names, threshold=85):
    updated = []
    df["Company Name"] = df["Company Name"].str.replace(r"\s*Ltd\.?", "", regex=True).str.strip()
    df["Company Name"] = df["Company Name"].str.replace(r"Bank", "", regex=True).str.strip()


    for original in df['Company Name']:
        match = match_company_name_fuzzy(original, reference_names, threshold)
        updated_name = match if match else original  # use original if no match
        updated.append(updated_name)

    df = df.copy()
    df['Company Name'] = updated
    return df

# 👇 Run this on df_unified
df_final = standardize_company_names(df_unified, reference_names, threshold = 85)

# # Optional: Save to file
# output_path = "/content/drive/MyDrive/df_financial_reports_q1_2021_full_names.csv"
# df_final.to_csv(output_path, index=False)
# print("✅ Cleaned DataFrame saved to:", output_path)


In [398]:
df_final

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,ACCEL,2025-04-16,NaN,NaN,0.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164755008.0,NaN,NaN
1,ACKERSTEIN GROUP,2025-04-16,9.023530e+08,123406000.0,0.430,2.501570e+08,1.104070e+08,NaN,NaN,NaN,1.944990e+09,NaN,NaN,7.484000e+07,NaN,NaN,288060992.0,NaN,NaN
2,Kvutzat Acro,2025-04-16,7.484870e+08,20589000.0,0.330,2.003150e+08,9.604400e+07,NaN,NaN,NaN,6.929554e+09,NaN,NaN,3.020661e+09,NaN,NaN,63043600.0,NaN,NaN
3,Adgar Investments and Development,2025-04-16,3.380300e+08,-7128000.0,-0.040,2.888100e+08,2.350480e+08,NaN,NaN,NaN,5.490045e+09,NaN,NaN,2.967060e+09,NaN,NaN,164856000.0,NaN,NaN
4,AERODROME GROUP,2025-04-16,1.445000e+07,-22798000.0,-0.250,9.900000e+05,-1.356500e+07,NaN,NaN,NaN,3.909100e+07,NaN,NaN,3.510000e+05,NaN,NaN,91569504.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,ZANLAKOL,2025-04-16,4.815950e+08,49839000.0,3.490,1.544380e+08,8.245000e+07,NaN,NaN,NaN,4.842290e+08,NaN,NaN,0.000000e+00,NaN,NaN,13991100.0,NaN,NaN
450,Zephyrus Wing Energies,2025-04-16,1.967430e+08,82409000.0,1.270,1.571700e+08,8.147100e+07,NaN,NaN,NaN,1.492863e+09,NaN,NaN,7.211830e+08,NaN,NaN,65017300.0,NaN,NaN
451,Z.M.H Hammerman,2025-04-16,3.900150e+08,39988000.0,2.000,4.439500e+07,3.852000e+06,NaN,NaN,NaN,1.805569e+09,NaN,NaN,2.282560e+08,NaN,NaN,20022000.0,NaN,NaN
452,ZOOZ POWER,2025-04-16,1.041000e+06,-10990000.0,-4.250,-4.860000e+05,-1.053600e+07,NaN,NaN,NaN,1.283700e+07,NaN,NaN,NaN,NaN,NaN,12105500.0,NaN,NaN


### Standardizing Company Names via Mapping and Fuzzy Variants

This stage refines company names by matching them to official names from a ticker mapping Excel file using both a manual override dictionary and a robust fuzzy matching system. It generates multiple name variants (e.g., with/without suffixes) and uses token-based similarity scoring to find the best match. This ensures alignment between report names and official tickers, which is crucial for downstream financial analysis.


In [399]:

# Load mapping Excel (already loaded earlier as mapping_df)
mapping_df = pd.read_excel(mapping_path)
mapping_df = mapping_df[['CompanyName', 'Ticker', 'JsonNames']].fillna("")

# Build alias dictionary: maps every known JSON name to the official CompanyName
json_name_to_real = {}

for _, row in mapping_df.iterrows():
    real_name = str(row['CompanyName']).strip()
    json_names = row['JsonNames']

    aliases = set()

    # Try parsing as a list of names
    if isinstance(json_names, str):
        try:
            parsed = ast.literal_eval(json_names)
            if isinstance(parsed, list):
                aliases.update([x.strip().upper() for x in parsed])
            else:
                aliases.add(json_names.strip().upper())
        except:
            aliases.add(json_names.strip().upper())

    # Add mapping from each alias to the official company name
    for alias in aliases:
        if alias:
            json_name_to_real[alias] = real_name.upper()

    # Also map the official name to itself
    json_name_to_real[real_name.upper()] = real_name.upper()

# Now apply the mapping to the unified DataFrame
unified_df = df_unified.copy()
unified_df['Company Name'] = unified_df['Company Name'].astype(str).str.strip()

standardized_rows = []
not_found = []

grouped = unified_df.groupby('Company Name')

for name, group in grouped:
    key = name.strip().upper()

    # Step 1: Try to find matching canonical name
    if key in json_name_to_real:
        real_name = json_name_to_real[key]
    else:
        not_found.append(name)
        standardized_rows.append(group.iloc[0])  # keep as-is
        continue

    # Step 2: Get all rows corresponding to this real name
    aliases = [k for k, v in json_name_to_real.items() if v == real_name]
    matching_rows = unified_df[unified_df['Company Name'].str.upper().isin(aliases)]

    # Step 3: Merge info – use row with fewest NaNs
    merged_row = matching_rows.loc[matching_rows.isna().sum(axis=1).idxmin()].copy()
    merged_row['Company Name'] = real_name
    standardized_rows.append(merged_row)

# Final result
df_final_cleaned = pd.DataFrame(standardized_rows).reset_index(drop=True)

df_final_cleaned


,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,AFI PROPERTIES,2025-04-16,1.498408e+09,663303000.0,17.430,9.781800e+08,8.515530e+08,NaN,NaN,NaN,2.143811e+10,NaN,NaN,1.025546e+10,NaN,NaN,38056400.0,NaN,NaN
1,ARGO Properties N.V.,2025-04-16,3.774700e+07,35184000.0,7.730,2.211600e+07,1.341600e+07,NaN,NaN,NaN,8.393710e+08,NaN,NaN,3.449680e+08,NaN,NaN,20709700.0,NaN,NaN
2,ACCEL,2025-04-16,NaN,NaN,0.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164755008.0,NaN,NaN
3,ACKERSTEIN GROUP,2025-04-16,9.023530e+08,123406000.0,0.430,2.501570e+08,1.104070e+08,NaN,NaN,NaN,1.944990e+09,NaN,NaN,7.484000e+07,NaN,NaN,288060992.0,NaN,NaN
4,ADGAR INVESTMENTS,2025-04-16,3.380300e+08,-7128000.0,-0.040,2.888100e+08,2.350480e+08,NaN,NaN,NaN,5.490045e+09,NaN,NaN,2.967060e+09,NaN,NaN,164856000.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,ZANLAKOL,2025-04-16,4.815950e+08,49839000.0,3.490,1.544380e+08,8.245000e+07,NaN,NaN,NaN,4.842290e+08,NaN,NaN,0.000000e+00,NaN,NaN,13991100.0,NaN,NaN
418,Zephyrus Wing Energies,2025-04-16,1.967430e+08,82409000.0,1.270,1.571700e+08,8.147100e+07,NaN,NaN,NaN,1.492863e+09,NaN,NaN,7.211830e+08,NaN,NaN,65017300.0,NaN,NaN
419,ZUR,2025-04-16,5.260541e+09,64328000.0,0.980,2.014652e+09,1.277634e+09,NaN,NaN,NaN,2.205779e+10,NaN,NaN,NaN,NaN,NaN,65099900.0,NaN,NaN
420,Zvi Sarfati & Sons Investments & Constructions,2025-04-16,4.822950e+08,51535000.0,2.970,1.419980e+08,1.035990e+08,NaN,NaN,NaN,1.379921e+09,NaN,NaN,1.084940e+08,NaN,NaN,17398200.0,NaN,NaN


In [400]:
# print("🔍 Companies not found in Excel mapping:")
# for name in not_found_unique:
#     print("-", name)


### Mapping JSON Aliases to Official Company Names

This block parses the `JsonNames` column from the Excel mapping file to extract known name aliases for each company. It builds a clear mapping from each alias to its official company name, helping resolve inconsistencies between report data and the reference ticker list. The result is printed in a readable format for verification.


In [401]:
# Create and print mapping: alias from 'JsonNames' → official company name
print("\n📌 JSON Alias Name → Official Company Name Mapping (from JsonNames column):\n")

alias_to_real = []

for _, row in mapping_df.iterrows():
    real_name = str(row['CompanyName']).strip()
    json_names = row['JsonNames']

    if isinstance(json_names, str) and json_names:
        try:
            parsed = ast.literal_eval(json_names)
            if isinstance(parsed, list):
                for alias in parsed:
                    alias = alias.strip()
                    if alias:
                        alias_to_real.append(f"'{alias}' --> '{real_name}'")
            else:
                alias = json_names.strip()
                if alias:
                    alias_to_real.append(f"'{alias}' --> '{real_name}'")
        except:
            alias = json_names.strip()
            if alias:
                alias_to_real.append(f"'{alias}' --> '{real_name}'")

# Sort alphabetically for readability
alias_to_real = sorted(set(alias_to_real))

for line in alias_to_real:
    print(line)



📌 JSON Alias Name → Official Company Name Mapping (from JsonNames column):

'"בזק" החברה הישראלית לתקשורת בע"מ' --> 'BEZEQ'
'(Y.Z) Queenco' --> 'QUEENCO'
'A. Luzon Real Estate and Finance' --> 'LUZON GROUP'
'A.I. Systems Conversation' --> 'AI SYSTEMS'
'A.S. Australia Israel Holdings' --> 'AUSTRALIA ISR'
'ALBAAD Massuot Yitzhak' --> 'ALBAAD'
'Abra Technologies Information' --> 'ABRA'
'Abra Technologies' --> 'ABRA'
'Accel Solutions Group' --> 'ACCEL'
'Adgar Investments & Development LTD.' --> 'ADGAR INVESTMENTS'
'Adgar Investments and Development' --> 'ADGAR INVESTMENTS'
'Afi Capital Nadlan' --> 'AFI PROPERTIES'
'Africa Israel Residences Ltd' --> 'AFRICA ISRAEL RESIDENCES'
'Airport City Ltd.' --> 'AIRPORT CITY'
'Airtouch Solar' --> 'AIRTOUCH'
'Alarum Technologies' --> 'ALARUM'
'Albad Masuot Yitzhak' --> 'ALBAAD'
'Allot Ltd.' --> 'ALLOT'
'Almeda Ventures Limited Partnership' --> 'ALMEDA PARTICIPATION UNIT'
'Almeda Ventures' --> 'ALMEDA PARTICIPATION UNIT'
'Almogim holdings' --> 'ALMOGIM'

In [402]:
df_final_cleaned

,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,AFI PROPERTIES,2025-04-16,1.498408e+09,663303000.0,17.430,9.781800e+08,8.515530e+08,NaN,NaN,NaN,2.143811e+10,NaN,NaN,1.025546e+10,NaN,NaN,38056400.0,NaN,NaN
1,ARGO Properties N.V.,2025-04-16,3.774700e+07,35184000.0,7.730,2.211600e+07,1.341600e+07,NaN,NaN,NaN,8.393710e+08,NaN,NaN,3.449680e+08,NaN,NaN,20709700.0,NaN,NaN
2,ACCEL,2025-04-16,NaN,NaN,0.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164755008.0,NaN,NaN
3,ACKERSTEIN GROUP,2025-04-16,9.023530e+08,123406000.0,0.430,2.501570e+08,1.104070e+08,NaN,NaN,NaN,1.944990e+09,NaN,NaN,7.484000e+07,NaN,NaN,288060992.0,NaN,NaN
4,ADGAR INVESTMENTS,2025-04-16,3.380300e+08,-7128000.0,-0.040,2.888100e+08,2.350480e+08,NaN,NaN,NaN,5.490045e+09,NaN,NaN,2.967060e+09,NaN,NaN,164856000.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,ZANLAKOL,2025-04-16,4.815950e+08,49839000.0,3.490,1.544380e+08,8.245000e+07,NaN,NaN,NaN,4.842290e+08,NaN,NaN,0.000000e+00,NaN,NaN,13991100.0,NaN,NaN
418,Zephyrus Wing Energies,2025-04-16,1.967430e+08,82409000.0,1.270,1.571700e+08,8.147100e+07,NaN,NaN,NaN,1.492863e+09,NaN,NaN,7.211830e+08,NaN,NaN,65017300.0,NaN,NaN
419,ZUR,2025-04-16,5.260541e+09,64328000.0,0.980,2.014652e+09,1.277634e+09,NaN,NaN,NaN,2.205779e+10,NaN,NaN,NaN,NaN,NaN,65099900.0,NaN,NaN
420,Zvi Sarfati & Sons Investments & Constructions,2025-04-16,4.822950e+08,51535000.0,2.970,1.419980e+08,1.035990e+08,NaN,NaN,NaN,1.379921e+09,NaN,NaN,1.084940e+08,NaN,NaN,17398200.0,NaN,NaN


### Alias Resolution and Final Deduplication

This final cleaning phase ensures that company names appearing as aliases (from the `JsonNames` column) are mapped to their official names using a pre-built alias dictionary. After resolving all aliases, the dataset is filtered to retain only companies listed in the Excel mapping. For each matched company, the most complete record (least missing values) is selected, resulting in a clean and deduplicated DataFrame ready for export.


In [403]:
# Step 0: Normalize alias_to_real into a usable map
# (Assuming alias_to_real was built earlier as a list of "'alias' --> 'real_name'" strings)
alias_dict = {}
for line in alias_to_real:
    try:
        alias, real = line.split("-->")
        alias = alias.strip().strip("'").upper()
        real = real.strip().strip("'").upper()
        alias_dict[alias] = real
    except:
        continue

# Step 1: Replace aliases in df_final_cleaned using alias_dict
df_with_resolved_aliases = df_final_cleaned.copy()
df_with_resolved_aliases['Company Name'] = df_with_resolved_aliases['Company Name'].str.upper().str.strip()
df_with_resolved_aliases['Company Name'] = df_with_resolved_aliases['Company Name'].apply(
    lambda name: alias_dict.get(name, name)
)

# Step 2: Get all real company names from Excel
real_company_names_from_excel = mapping_df['CompanyName'].dropna().str.upper().unique().tolist()

# Step 3: Filter to include only companies that match Excel names after alias resolution
filtered_df = df_with_resolved_aliases[
    df_with_resolved_aliases['Company Name'].isin(real_company_names_from_excel)
].copy()

# Step 4: For each company, keep the row with the fewest NaNs
best_rows = []
for company_name in filtered_df['Company Name'].unique():
    group = filtered_df[filtered_df['Company Name'] == company_name]
    best_row = group.loc[group.isna().sum(axis=1).idxmin()]
    best_rows.append(best_row)

# Step 5: Final deduplicated DataFrame
df_final_deduped = pd.DataFrame(best_rows).reset_index(drop=True)

# Show result
print("✅ Deduplicated DataFrame shape:", df_final_deduped.shape)
df_final_deduped


✅ Deduplicated DataFrame shape: (196, 19)


,Company Name,Report Date,Revenue,Net Income,Earnings Per Share (EPS),Gross Profit,Operating Income (EBIT),Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Total Assets,Total Liabilities,Short-Term Debt,Long-Term Debt,Shareholders’ Equity,Dividends Paid,Number of Outstanding Shares,Guidance/Forecast,Key Performance Indicators (KPIs)
0,AFI PROPERTIES,2025-04-16,1.498408e+09,663303000.0,17.430,9.781800e+08,8.515530e+08,NaN,NaN,NaN,2.143811e+10,NaN,NaN,1.025546e+10,NaN,NaN,38056400.0,NaN,NaN
1,ACCEL,2025-04-16,NaN,NaN,0.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164755008.0,NaN,NaN
2,ACKERSTEIN GROUP,2025-04-16,9.023530e+08,123406000.0,0.430,2.501570e+08,1.104070e+08,NaN,NaN,NaN,1.944990e+09,NaN,NaN,7.484000e+07,NaN,NaN,288060992.0,NaN,NaN
3,ADGAR INVESTMENTS,2025-04-16,3.380300e+08,-7128000.0,-0.040,2.888100e+08,2.350480e+08,NaN,NaN,NaN,5.490045e+09,NaN,NaN,2.967060e+09,NaN,NaN,164856000.0,NaN,NaN
4,AERODROME GROUP,2025-04-16,1.445000e+07,-22798000.0,-0.250,9.900000e+05,-1.356500e+07,NaN,NaN,NaN,3.909100e+07,NaN,NaN,3.510000e+05,NaN,NaN,91569504.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,XTL BIO,2025-04-16,NaN,NaN,-0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,881385024.0,NaN,NaN
192,YBOX,2025-04-16,4.782100e+07,-9689000.0,-0.030,7.792000e+06,-1.296800e+07,NaN,NaN,NaN,1.454893e+09,NaN,NaN,5.013450e+08,NaN,NaN,355999008.0,NaN,NaN
193,ZOOZ POWER,2025-04-16,1.041000e+06,-10990000.0,-4.250,-4.860000e+05,-1.053600e+07,NaN,NaN,NaN,1.283700e+07,NaN,NaN,NaN,NaN,NaN,12105500.0,NaN,NaN
194,ZANLAKOL,2025-04-16,4.815950e+08,49839000.0,3.490,1.544380e+08,8.245000e+07,NaN,NaN,NaN,4.842290e+08,NaN,NaN,0.000000e+00,NaN,NaN,13991100.0,NaN,NaN


### Exporting to drive

In [404]:
df_final_deduped.to_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/report_analysis_2021_q1.csv")